- r7.0_fr0_jv7_default                          37        50            10             370   ERROR
- r7.1_fr0_jv7_repaironly                       39        50            10             390   ERROR
- r7.2_fr0_jv7_soltaper_0.015                   44        50            10             440   ERROR
- r7.3_fr0_jv7_soltaper_0.060                   41        50            10             410   ERROR
- r7.4_fr0_jv7_pftaper_0.005                    26        50            10             260   ERROR
- r7.5_fr0_jv7_pftaper_0.020                    38        50            10             380   ERROR
- r7.6_fr0_jv7_legtaper                         50        50            10             500   COMPLETE
- r7.7_fr0_jv7_blend_0.95-0.995                 38        50            10             380   ERROR
- r7.8_fr0_jv7_blend_0.97-0.99                  39        50            10             390   ERROR
- r7.9_fr0_jv6_control                          40        50            10             400   ERROR

In [3]:
from pathlib import Path
RUN_DIR = "r9.0_fr0_multin_zonal_seedall"
grid_fname= Path(f"{RUN_DIR}/d3d_r8.0_pfflat.grd.nc") # Path(f"{RUN_DIR}/d3d_shelf_184016.03506_436_nc3.grd.nc") # 

In [4]:
import sys
sys.path.append("/global/cfs/projectdirs/m4491/btaczak/SOLPS-BOUT-coupling/scripts")
import os
import sys
import yaml
import numpy as np
import re
import numpy as np
import netCDF4 as nc
import yaml
from scipy.interpolate import interp1d
from boutdata.data import BoutData
from boututils.datafile import DataFile
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter
from boutdata.data import BoutData
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, fixed
from sibil_io_functions import load_config, sibil_config_summary, confirm, parse_tdata2, parse_block_sizes, write_updated_namelist, _format_value, extract_tdata_first_index, interpolate_dataset, get_midplane_distances_from_psin, vec_summary, set_tdata2_block, get_bout_midplane_distances
from converged_profile_functions import unbiased_convergence, end_biased_convergence

!echo $PWD
data_dir  = f"{RUN_DIR}/numpy"
grid_fname = str(grid_fname)
transport_coeffs = ["Da_t", "Xe_t", "Xi_t", "Vr_mean_txy", "Vexbx_txy", "Vbtildx", "ddr_Tetot_txy", "ddr_Titot_txy", "ddr_nitot_txy"]
jomp = 44
N, l = 0,7

grid = DataFile(grid_fname)
try:
    _ = grid["psi_bdry"]
    has_psi_bdry = True
except Exception:
    has_psi_bdry = False
    
if has_psi_bdry:
    psi_bndry = grid["psi_bdry"]
else:
    psi_bndry = grid["psi_bndry"]

psin = ((grid["psixy"] - grid["psi_axis"]) / (psi_bndry - grid["psi_axis"]))[:,jomp]

psin_mid, sepx_dists_bout = get_bout_midplane_distances(grid_fname)

convergent_transport_coeffs = {}
interplted_transport_coeffs = {}
transport_coeffs = transport_coeffs

for var_name in transport_coeffs:
    if "ddr" in var_name:
        print(var_name)
        convergent_transport_coeffs[var_name] = 1/np.load(f"{data_dir}/{var_name}.npy")
        continue
    var = np.load(f"{data_dir}/{var_name}.npy")
    var_conv = unbiased_convergence(var, None, None)
    convergent_transport_coeffs[var_name] = var_conv

import sibil_debug_diagnostics as dbg
!echo $PWD
display(dbg.make_debug_widget(convergent_transport_coeffs, psin_mid,
                              jomp=jomp, grid=grid, data_dir=data_dir))
dbg.full_report(convergent_transport_coeffs, psin_mid, grid_fname, jomp=jomp,
                savedir=f"./crash_diagnostics/{RUN_DIR}", run=RUN_DIR)
!echo $PWD

/global/cfs/cdirs/m4491/btaczak/bout-runs/pfr_tests/r9_scan/runs


FileNotFoundError: [Errno 2] No such file or directory: 'r9.0_fr0_multin_zonal_seedall/numpy/Da_t.npy'

import matplotlib.pyplot as plt
import boututils.efit_analyzer as ea
from boututils.read_geqdsk import read_geqdsk

ea.show = lambda *args, **kwargs: None

g = read_geqdsk("/global/cfs/projectdirs/m4491/btaczak/SIBIL-runs/DIIID_LSN_noC_corrected_heat_flux_test/g184016.03200")
ea.View2D(g, option=0)
plt.show()